In [10]:
# ============================================================
# NOTEBOOK 4: HEALTH RISK RECOMMENDATIONS
# Purpose: Take patient details, predict disease risks,
#          and generate fitness/diet recommendations
# ============================================================

import pandas as pd
import numpy as np
import joblib
from pathlib import Path

BASE_DIR    = Path(r"D:\iit\DSGP\NutriScanner\health-risk-recommendations\research\NoteBooks")
MODEL_DIR   = BASE_DIR / "models"

DISEASE_COLUMNS = [
    "Diabetes_Risk", "Hypertension_Risk", "Heart_Disease_Risk",
    "Obesity_Risk",  "Anemia_Risk",       "Kidney_Disease_Risk"
]

BASE_FEATURES = [
    "Age", "Gender", "BMI", "Daily_Calories_kcal", "Carbohydrates_g",
    "Protein_g", "Total_Fat_g", "Saturated_Fat_g", "Trans_Fat_g",
    "Total_Sugar_g", "Added_Sugar_g", "Fiber_g", "Sodium_mg", "Potassium_mg",
    "Calcium_mg", "Iron_mg", "Vitamin_D_IU", "Vitamin_B12_mcg",
    "Physical_Activity_min", "Water_Intake_L"
]

DISEASE_FEATURES = {
    "Diabetes_Risk":       [f for f in BASE_FEATURES if f not in ["Added_Sugar_g", "Total_Sugar_g"]],
    "Hypertension_Risk":   [f for f in BASE_FEATURES if f not in ["Sodium_mg", "Potassium_mg"]],
    "Heart_Disease_Risk":  [f for f in BASE_FEATURES if f not in ["Saturated_Fat_g", "Trans_Fat_g", "Total_Fat_g"]],
    "Obesity_Risk":        [f for f in BASE_FEATURES if f not in ["BMI", "Daily_Calories_kcal"]],
    "Anemia_Risk":         [f for f in BASE_FEATURES if f not in ["Iron_mg", "Vitamin_B12_mcg"]],
    "Kidney_Disease_Risk": [f for f in BASE_FEATURES if f not in ["Sodium_mg", "Water_Intake_L", "Protein_g"]]
}

models = {}
for disease in DISEASE_COLUMNS:
    models[disease] = joblib.load(MODEL_DIR / f"{disease}_model.pkl")

print("All models loaded ✅")

All models loaded ✅


In [14]:
# ============================================================
# CELL 2: HELPER FUNCTIONS
# ============================================================

def calculate_bmi(weight_kg, height_cm):
    return round(weight_kg / (height_cm / 100) ** 2, 1)

def bmi_obesity_level(bmi):
    if bmi < 18.5:   return "Underweight",     "🟢", "LOW"
    elif bmi < 25.0: return "Normal Weight",   "🟢", "LOW"
    elif bmi < 30.0: return "Overweight",      "🟡", "MODERATE"
    elif bmi < 35.0: return "Obese Class I",   "🔴", "HIGH"
    elif bmi < 40.0: return "Obese Class II",  "🔴", "HIGH"
    else:            return "Obese Class III", "🔴", "HIGH"

def get_risk_level(prob):
    if prob >= 0.70:   return "HIGH",     "🔴"
    elif prob >= 0.40: return "MODERATE", "🟡"
    else:              return "LOW",      "🟢"

def combined_diabetes_risk(model_prob, total_sugar_g, added_sugar_g, bmi):
    if added_sugar_g > 60 or total_sugar_g > 100: rule_prob = 0.85
    elif added_sugar_g > 40 or total_sugar_g > 70: rule_prob = 0.65
    elif added_sugar_g < 10 and total_sugar_g < 30: rule_prob = 0.08
    else: rule_prob = 0.35
    if bmi >= 30: rule_prob = min(rule_prob + 0.10, 0.98)
    return round((0.5 * model_prob) + (0.5 * rule_prob), 3)

def combined_hypertension_risk(model_prob, sodium_mg, potassium_mg):
    if sodium_mg > 4000 and potassium_mg < 1500: rule_prob = 0.85
    elif sodium_mg > 3000: rule_prob = 0.60
    elif sodium_mg < 1500 and potassium_mg > 3000: rule_prob = 0.10
    else: rule_prob = 0.35
    return round((0.5 * model_prob) + (0.5 * rule_prob), 3)

def combined_heart_risk(model_prob, saturated_fat_g, trans_fat_g, total_fat_g):
    if trans_fat_g > 3.5 or saturated_fat_g > 50: rule_prob = 0.88
    elif trans_fat_g > 2.0 or saturated_fat_g > 30: rule_prob = 0.62
    elif saturated_fat_g < 15 and trans_fat_g < 1.0: rule_prob = 0.10
    else: rule_prob = 0.35
    return round((0.5 * model_prob) + (0.5 * rule_prob), 3)

def combined_anemia_risk(model_prob, iron_mg, b12_mcg):
    if iron_mg < 5 or b12_mcg < 1.0: rule_prob = 0.90
    elif iron_mg < 8 or b12_mcg < 2.0: rule_prob = 0.60
    elif iron_mg > 15 and b12_mcg > 3.0: rule_prob = 0.08
    else: rule_prob = 0.30
    return round((0.5 * model_prob) + (0.5 * rule_prob), 3)

def combined_kidney_risk(model_prob, sodium_mg, protein_g, water_intake_l):
    if sodium_mg > 4500 and protein_g > 130: rule_prob = 0.88
    elif sodium_mg > 3500 or protein_g > 110: rule_prob = 0.62
    elif water_intake_l < 0.8: rule_prob = 0.55
    elif sodium_mg < 1500 and protein_g < 80 and water_intake_l > 2.5: rule_prob = 0.08
    else: rule_prob = 0.30
    return round((0.5 * model_prob) + (0.5 * rule_prob), 3)

print("Helper functions loaded ✅")

Helper functions loaded ✅


In [15]:
# ============================================================
# CELL 3: RECOMMENDATION ENGINE
# Gives fitness + diet tips only for HIGH or MODERATE risks
# ============================================================

RECOMMENDATIONS = {

    "Obesity": {
        "HIGH": {
            "fitness": [
                "Do at least 45–60 min of cardio daily (brisk walking, cycling, swimming)",
                "Add 3x/week strength training to build metabolism-boosting muscle",
                "Avoid prolonged sitting — stand or walk every 30 minutes",
                "Track your daily steps — aim for 10,000+ steps per day",
                "Consider a structured weight-loss program with a fitness coach"
            ],
            "diet": [
                "Reduce daily calorie intake by 500–700 kcal from your current level",
                "Cut out processed foods, fried items, and sugary beverages entirely",
                "Fill half your plate with vegetables at every meal",
                "Eat smaller portions — use a smaller plate to control serving size",
                "Avoid eating after 8 PM and skip late-night snacking"
            ]
        },
        "MODERATE": {
            "fitness": [
                "Aim for 30 min of moderate exercise at least 5 days a week",
                "Include light strength exercises 2x/week",
                "Take the stairs and walk instead of driving for short distances"
            ],
            "diet": [
                "Reduce sugary drinks and replace with water or herbal teas",
                "Increase fiber intake — add more vegetables, legumes, and whole grains",
                "Watch portion sizes and avoid second servings"
            ]
        }
    },

    "Diabetes": {
        "HIGH": {
            "fitness": [
                "Walk for 30 minutes after every main meal to lower blood sugar spikes",
                "Do resistance training 3x/week — it significantly improves insulin sensitivity",
                "Avoid high-intensity exercise on an empty stomach",
                "Monitor blood sugar before and after workouts",
                "Aim for 150+ min of moderate aerobic activity per week"
            ],
            "diet": [
                "Eliminate added sugars and sweetened drinks completely",
                "Switch to low-GI carbohydrates — oats, brown rice, sweet potato",
                "Eat every 3–4 hours in small meals to avoid blood sugar spikes",
                "Increase dietary fiber to slow glucose absorption — target 25–35g/day",
                "Limit white bread, white rice, and refined cereals"
            ]
        },
        "MODERATE": {
            "fitness": [
                "Walk at least 30 min daily, preferably after meals",
                "Include light resistance training 2x/week",
                "Stay active throughout the day — avoid long sitting periods"
            ],
            "diet": [
                "Reduce added sugar intake — cut sweets, pastries, and sodas",
                "Choose whole grain options over refined carbohydrates",
                "Include more vegetables and legumes in daily meals"
            ]
        }
    },

    "Hypertension": {
        "HIGH": {
            "fitness": [
                "Do 30–45 min of moderate cardio daily — walking, cycling, or swimming",
                "Avoid heavy weight lifting as it can spike blood pressure temporarily",
                "Practice breathing exercises and yoga for stress-related BP control",
                "Exercise at a conversational pace — you should be able to talk while active",
                "Check blood pressure before and after workouts"
            ],
            "diet": [
                "Strictly limit sodium intake to under 1,500 mg per day",
                "Follow the DASH diet — rich in fruits, vegetables, and low-fat dairy",
                "Increase potassium intake — bananas, spinach, sweet potatoes, avocados",
                "Avoid processed meats, canned foods, and fast food (all high in sodium)",
                "Limit alcohol and caffeine consumption"
            ]
        },
        "MODERATE": {
            "fitness": [
                "Walk briskly for 30 minutes at least 5 days a week",
                "Include yoga or stretching for stress reduction",
                "Avoid extreme exertion and monitor your heart rate during exercise"
            ],
            "diet": [
                "Reduce sodium — avoid adding extra salt to meals",
                "Eat more potassium-rich foods like bananas and leafy greens",
                "Cut down on processed and packaged foods"
            ]
        }
    },

    "Heart Disease": {
        "HIGH": {
            "fitness": [
                "Start with low-impact cardio — 20–30 min walking, swimming, or cycling",
                "Avoid sudden bursts of intense exercise — build intensity gradually",
                "Do light stretching and yoga to reduce cardiovascular stress",
                "Exercise 5 days a week at moderate intensity — never push to exhaustion",
                "Consult a cardiologist before starting any new exercise program"
            ],
            "diet": [
                "Eliminate trans fats completely — check all food labels",
                "Reduce saturated fat — avoid butter, fatty meats, and full-fat dairy",
                "Eat oily fish 2–3x per week for omega-3 fatty acids (salmon, sardines)",
                "Increase soluble fiber — oats, flaxseed, apples, and lentils",
                "Replace refined oils with olive oil or avocado oil for cooking"
            ]
        },
        "MODERATE": {
            "fitness": [
                "Aim for 30 min of light to moderate cardio 5x/week",
                "Avoid smoking and limit alcohol — both directly affect heart health",
                "Include stress-reducing activities like meditation or yoga"
            ],
            "diet": [
                "Reduce saturated fat intake — choose lean meats and low-fat dairy",
                "Avoid deep-fried and heavily processed foods",
                "Add more fruits, vegetables, and whole grains to your daily diet"
            ]
        }
    },

    "Anemia": {
        "HIGH": {
            "fitness": [
                "Avoid high-intensity exercise until iron/B12 levels are restored",
                "Stick to gentle activity — light walking or yoga",
                "Rest when fatigued — pushing through anemia fatigue can cause injury",
                "Gradually increase exercise intensity as your levels improve",
                "Inform your trainer or doctor about your anemia before exercising"
            ],
            "diet": [
                "Eat iron-rich foods daily — red meat, lentils, spinach, tofu, fortified cereals",
                "Pair iron-rich foods with Vitamin C (lemon juice, oranges) to boost absorption",
                "Take Vitamin B12 through eggs, dairy, fish, or supplements if needed",
                "Avoid tea or coffee immediately after meals — they block iron absorption",
                "Consider iron and B12 supplements after consulting a doctor"
            ]
        },
        "MODERATE": {
            "fitness": [
                "Keep exercise light to moderate — listen to your body",
                "Include short walks and light stretching daily",
                "Avoid overexertion when feeling tired or breathless"
            ],
            "diet": [
                "Increase iron intake — add lentils, spinach, and lean meat to meals",
                "Add Vitamin C to every meal to enhance iron absorption",
                "Include B12 sources — eggs, dairy, fish, or fortified foods"
            ]
        }
    },

    "Kidney Disease": {
        "HIGH": {
            "fitness": [
                "Keep exercise gentle — walking and stretching are safest",
                "Avoid dehydration during exercise — drink water before, during, and after",
                "Do not do high-impact or strenuous workouts without medical clearance",
                "Exercise 3–4x per week at a light to moderate pace",
                "Consult a nephrologist before starting any fitness program"
            ],
            "diet": [
                "Drastically reduce sodium — stay well under 1,500 mg per day",
                "Limit protein intake to reduce kidney filtration load",
                "Drink at least 2–3 liters of water daily to flush kidneys",
                "Avoid high-potassium foods if advised by your doctor (bananas, tomatoes)",
                "Cut out processed foods, canned goods, and fast food entirely"
            ]
        },
        "MODERATE": {
            "fitness": [
                "Walk 20–30 min daily and stay consistently hydrated",
                "Avoid heavy weightlifting that strains the body",
                "Include gentle yoga or stretching for overall wellness"
            ],
            "diet": [
                "Reduce sodium and processed food consumption",
                "Drink more water throughout the day — at least 2 liters",
                "Monitor protein intake — avoid very high-protein diets"
            ]
        }
    }
}

print("Recommendation engine loaded ✅")

Recommendation engine loaded ✅


In [16]:
# ============================================================
# CELL 4: PREDICT + RECOMMEND FOR A SINGLE PATIENT
# ============================================================

def predict_and_recommend(patient):

    name   = patient["name"]
    bmi    = calculate_bmi(patient["weight_kg"], patient["height_cm"])
    patient["BMI"] = bmi
    bmi_cat, ob_icon, ob_level = bmi_obesity_level(bmi)

    # ── Collect all risk levels ──────────────────────────────────────────────
    risk_results = {}

    # Obesity — WHO only
    risk_results["Obesity"] = (None, ob_icon, ob_level, bmi_cat)

    # Other diseases — model + rule
    disease_map = {
        "Diabetes_Risk": lambda p: combined_diabetes_risk(
            models["Diabetes_Risk"].predict_proba(
                pd.DataFrame([{f: p[f] for f in DISEASE_FEATURES["Diabetes_Risk"]}]))[0][1],
            p["Total_Sugar_g"], p["Added_Sugar_g"], bmi),

        "Hypertension_Risk": lambda p: combined_hypertension_risk(
            models["Hypertension_Risk"].predict_proba(
                pd.DataFrame([{f: p[f] for f in DISEASE_FEATURES["Hypertension_Risk"]}]))[0][1],
            p["Sodium_mg"], p["Potassium_mg"]),

        "Heart_Disease_Risk": lambda p: combined_heart_risk(
            models["Heart_Disease_Risk"].predict_proba(
                pd.DataFrame([{f: p[f] for f in DISEASE_FEATURES["Heart_Disease_Risk"]}]))[0][1],
            p["Saturated_Fat_g"], p["Trans_Fat_g"], p["Total_Fat_g"]),

        "Anemia_Risk": lambda p: combined_anemia_risk(
            models["Anemia_Risk"].predict_proba(
                pd.DataFrame([{f: p[f] for f in DISEASE_FEATURES["Anemia_Risk"]}]))[0][1],
            p["Iron_mg"], p["Vitamin_B12_mcg"]),

        "Kidney_Disease_Risk": lambda p: combined_kidney_risk(
            models["Kidney_Disease_Risk"].predict_proba(
                pd.DataFrame([{f: p[f] for f in DISEASE_FEATURES["Kidney_Disease_Risk"]}]))[0][1],
            p["Sodium_mg"], p["Protein_g"], p["Water_Intake_L"])
    }

    short_names = {
        "Diabetes_Risk":      "Diabetes",
        "Hypertension_Risk":  "Hypertension",
        "Heart_Disease_Risk": "Heart Disease",
        "Anemia_Risk":        "Anemia",
        "Kidney_Disease_Risk":"Kidney Disease"
    }

    for disease, fn in disease_map.items():
        prob          = fn(patient)
        level, icon   = get_risk_level(prob)
        short         = short_names[disease]
        risk_results[short] = (prob, icon, level, None)

    # ── Print Patient Header ─────────────────────────────────────────────────
    print(f"\n╔{'═'*62}╗")
    print(f"║  👤  {name:<56}║")
    print(f"╠{'═'*62}╣")
    print(f"║  Height : {patient['height_cm']} cm    "
          f"Weight : {patient['weight_kg']} kg    "
          f"BMI : {bmi:<14}║")
    print(f"╚{'═'*62}╝")

    # ── Print Obesity Block ──────────────────────────────────────────────────
    print(f"\n  ── Obesity Risk (WHO BMI Classification) {'─'*21}")
    print(f"     BMI        :  {bmi}")
    print(f"     Category   :  {bmi_cat}")
    print(f"     Risk Level :  {ob_icon}  {ob_level}")

    # ── Print Other Diseases ─────────────────────────────────────────────────
    print(f"\n  ── Other Disease Risks {'─'*39}")
    print(f"  {'Disease':<22}  {'Probability':>11}  {'Risk Level'}")
    print(f"  {'─'*22}  {'─'*11}  {'─'*15}")

    for name_key, (prob, icon, level, _) in risk_results.items():
        if name_key == "Obesity":
            continue
        print(f"  {name_key:<22}  {prob:>10.1%}  {icon}  {level}")

    # ── Print Recommendations ────────────────────────────────────────────────
    any_rec = False

    for disease_name, (prob, icon, level, extra) in risk_results.items():
        if level not in ("HIGH", "MODERATE"):
            continue
        if disease_name not in RECOMMENDATIONS:
            continue
        if level not in RECOMMENDATIONS[disease_name]:
            continue

        if not any_rec:
            print(f"\n  {'═'*62}")
            print(f"  RECOMMENDATIONS")
            print(f"  {'═'*62}")
            any_rec = True

        recs   = RECOMMENDATIONS[disease_name][level]
        header = f"  {icon}  {disease_name.upper()}  —  {level} RISK"

        print(f"\n{header}")
        print(f"  {'─'*60}")
        print(f"  Fitness:")
        for tip in recs["fitness"]:
            print(f"    •  {tip}")
        print(f"\n  Diet:")
        for tip in recs["diet"]:
            print(f"    •  {tip}")

    if not any_rec:
        print(f"\n  {'═'*62}")
        print(f"  ✅  All risks are LOW — keep up the healthy lifestyle!")
        print(f"  {'═'*62}")

    print(f"\n{'─'*64}\n")


print("predict_and_recommend() function ready ✅")

predict_and_recommend() function ready ✅


In [17]:
# ============================================================
# CELL 5: RUN ON ALL 10 DEMO PATIENTS
# ============================================================

demo_patients = [
    {
        "name": "P01 — Young healthy male",
        "height_cm": 175, "weight_kg": 66, "Age": 25, "Gender": 1,
        "Daily_Calories_kcal": 2100, "Carbohydrates_g": 230, "Protein_g": 110,
        "Total_Fat_g": 60, "Saturated_Fat_g": 12, "Trans_Fat_g": 0.3,
        "Total_Sugar_g": 30, "Added_Sugar_g": 8, "Fiber_g": 30,
        "Sodium_mg": 1300, "Potassium_mg": 3800, "Calcium_mg": 1100,
        "Iron_mg": 18, "Vitamin_D_IU": 900, "Vitamin_B12_mcg": 5.0,
        "Physical_Activity_min": 180, "Water_Intake_L": 3.2
    },
    {
        "name": "P02 — Middle-aged obese male",
        "height_cm": 170, "weight_kg": 108, "Age": 48, "Gender": 1,
        "Daily_Calories_kcal": 3600, "Carbohydrates_g": 390, "Protein_g": 85,
        "Total_Fat_g": 160, "Saturated_Fat_g": 58, "Trans_Fat_g": 4.2,
        "Total_Sugar_g": 115, "Added_Sugar_g": 70, "Fiber_g": 7,
        "Sodium_mg": 4600, "Potassium_mg": 1100, "Calcium_mg": 480,
        "Iron_mg": 7, "Vitamin_D_IU": 90, "Vitamin_B12_mcg": 1.1,
        "Physical_Activity_min": 5, "Water_Intake_L": 0.9
    },
    {
        "name": "P03 — Elderly diabetic female",
        "height_cm": 160, "weight_kg": 77, "Age": 68, "Gender": 0,
        "Daily_Calories_kcal": 2700, "Carbohydrates_g": 310, "Protein_g": 70,
        "Total_Fat_g": 95, "Saturated_Fat_g": 38, "Trans_Fat_g": 2.8,
        "Total_Sugar_g": 105, "Added_Sugar_g": 60, "Fiber_g": 9,
        "Sodium_mg": 3100, "Potassium_mg": 1700, "Calcium_mg": 550,
        "Iron_mg": 8, "Vitamin_D_IU": 140, "Vitamin_B12_mcg": 1.4,
        "Physical_Activity_min": 12, "Water_Intake_L": 1.1
    },
    {
        "name": "P04 — Fit young female",
        "height_cm": 165, "weight_kg": 55, "Age": 22, "Gender": 0,
        "Daily_Calories_kcal": 1950, "Carbohydrates_g": 210, "Protein_g": 95,
        "Total_Fat_g": 55, "Saturated_Fat_g": 10, "Trans_Fat_g": 0.2,
        "Total_Sugar_g": 28, "Added_Sugar_g": 6, "Fiber_g": 32,
        "Sodium_mg": 1200, "Potassium_mg": 4000, "Calcium_mg": 1200,
        "Iron_mg": 20, "Vitamin_D_IU": 1000, "Vitamin_B12_mcg": 5.5,
        "Physical_Activity_min": 200, "Water_Intake_L": 3.5
    },
    {
        "name": "P05 — Anemic underweight female",
        "height_cm": 162, "weight_kg": 45, "Age": 30, "Gender": 0,
        "Daily_Calories_kcal": 1400, "Carbohydrates_g": 170, "Protein_g": 38,
        "Total_Fat_g": 42, "Saturated_Fat_g": 9, "Trans_Fat_g": 0.6,
        "Total_Sugar_g": 40, "Added_Sugar_g": 18, "Fiber_g": 12,
        "Sodium_mg": 3900, "Potassium_mg": 1600, "Calcium_mg": 380,
        "Iron_mg": 3, "Vitamin_D_IU": 180, "Vitamin_B12_mcg": 0.7,
        "Physical_Activity_min": 25, "Water_Intake_L": 0.7
    },
    {
        "name": "P06 — Hypertension risk male",
        "height_cm": 172, "weight_kg": 87, "Age": 52, "Gender": 1,
        "Daily_Calories_kcal": 3100, "Carbohydrates_g": 270, "Protein_g": 92,
        "Total_Fat_g": 115, "Saturated_Fat_g": 42, "Trans_Fat_g": 3.0,
        "Total_Sugar_g": 65, "Added_Sugar_g": 32, "Fiber_g": 11,
        "Sodium_mg": 5200, "Potassium_mg": 1000, "Calcium_mg": 510,
        "Iron_mg": 10, "Vitamin_D_IU": 220, "Vitamin_B12_mcg": 2.2,
        "Physical_Activity_min": 18, "Water_Intake_L": 1.3
    },
    {
        "name": "P07 — Moderate risk elderly male",
        "height_cm": 168, "weight_kg": 75, "Age": 65, "Gender": 1,
        "Daily_Calories_kcal": 2300, "Carbohydrates_g": 240, "Protein_g": 78,
        "Total_Fat_g": 85, "Saturated_Fat_g": 28, "Trans_Fat_g": 1.8,
        "Total_Sugar_g": 58, "Added_Sugar_g": 28, "Fiber_g": 16,
        "Sodium_mg": 2600, "Potassium_mg": 2100, "Calcium_mg": 700,
        "Iron_mg": 11, "Vitamin_D_IU": 310, "Vitamin_B12_mcg": 2.6,
        "Physical_Activity_min": 30, "Water_Intake_L": 1.9
    },
    {
        "name": "P08 — Kidney risk male",
        "height_cm": 174, "weight_kg": 84, "Age": 55, "Gender": 1,
        "Daily_Calories_kcal": 2900, "Carbohydrates_g": 260, "Protein_g": 155,
        "Total_Fat_g": 100, "Saturated_Fat_g": 35, "Trans_Fat_g": 2.2,
        "Total_Sugar_g": 55, "Added_Sugar_g": 25, "Fiber_g": 13,
        "Sodium_mg": 4800, "Potassium_mg": 1300, "Calcium_mg": 590,
        "Iron_mg": 9, "Vitamin_D_IU": 250, "Vitamin_B12_mcg": 2.4,
        "Physical_Activity_min": 20, "Water_Intake_L": 0.6
    },
    {
        "name": "P09 — Teenage male borderline",
        "height_cm": 178, "weight_kg": 77, "Age": 17, "Gender": 1,
        "Daily_Calories_kcal": 2600, "Carbohydrates_g": 295, "Protein_g": 88,
        "Total_Fat_g": 88, "Saturated_Fat_g": 30, "Trans_Fat_g": 2.5,
        "Total_Sugar_g": 88, "Added_Sugar_g": 55, "Fiber_g": 13,
        "Sodium_mg": 2900, "Potassium_mg": 2000, "Calcium_mg": 820,
        "Iron_mg": 13, "Vitamin_D_IU": 380, "Vitamin_B12_mcg": 3.0,
        "Physical_Activity_min": 60, "Water_Intake_L": 2.0
    },
    {
        "name": "P10 — Heart disease risk female",
        "height_cm": 158, "weight_kg": 77, "Age": 60, "Gender": 0,
        "Daily_Calories_kcal": 2800, "Carbohydrates_g": 255, "Protein_g": 72,
        "Total_Fat_g": 140, "Saturated_Fat_g": 60, "Trans_Fat_g": 4.5,
        "Total_Sugar_g": 78, "Added_Sugar_g": 42, "Fiber_g": 8,
        "Sodium_mg": 3400, "Potassium_mg": 1600, "Calcium_mg": 520,
        "Iron_mg": 7, "Vitamin_D_IU": 160, "Vitamin_B12_mcg": 1.6,
        "Physical_Activity_min": 8, "Water_Intake_L": 1.0
    }
]

for patient in demo_patients:
    predict_and_recommend(patient)



╔══════════════════════════════════════════════════════════════╗
║  👤  P01 — Young healthy male                                ║
╠══════════════════════════════════════════════════════════════╣
║  Height : 175 cm    Weight : 66 kg    BMI : 21.6          ║
╚══════════════════════════════════════════════════════════════╝

  ── Obesity Risk (WHO BMI Classification) ─────────────────────
     BMI        :  21.6
     Category   :  Normal Weight
     Risk Level :  🟢  LOW

  ── Other Disease Risks ───────────────────────────────────────
  Disease                 Probability  Risk Level
  ──────────────────────  ───────────  ───────────────
  Diabetes                     18.4%  🟢  LOW
  Hypertension                 16.6%  🟢  LOW
  Heart Disease                 5.2%  🟢  LOW
  Anemia                        6.4%  🟢  LOW
  Kidney Disease               19.8%  🟢  LOW

  ══════════════════════════════════════════════════════════════
  ✅  All risks are LOW — keep up the healthy lifestyle!
  ═════════